In [8]:
import os
import numpy as np
import pandas as pd
import json
import copy
import random
from tqdm.auto import tqdm

import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth

import torch
import json
from transformers import AutoModelForCausalLM, AutoTokenizer, StoppingCriteria, StoppingCriteriaList, set_seed

ssid = 1241
random.seed(ssid)
np.random.seed(ssid)
torch.manual_seed(ssid)
torch.cuda.manual_seed_all(ssid)
set_seed(ssid)

In [2]:
init_dataset = pd.read_json("hf://datasets/Team-ACE/ToolACE/data.json")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## Parser functions

In [3]:
def tool_responses_to_text(responses):
    res = ""
    for response in responses:
        res += f"{response['name']}: {json.dumps(response['results'])}"
    return res

def parse_one_conversation(conv):
    res = []
    N = len(conv)
    i = 0
    while i < N:
        if conv[i]['from'] == 'tool':
            context = copy.deepcopy(json.loads(conv[i]['value']))
            new_entry = {}

            # Looking for the user request
            j = i
            while j >= 0:
                if conv[j]['from'] != 'user':
                    j -= 1
                else:
                    break
            else:
                print("Could not find user request???? Skipping the conversation...")
                return res
            new_entry['query'] = conv[j]['value']

            # Looking for the model response (it is absent in some conversations)
            j = i + 1
            while j < N:
                if conv[j]['from'] != 'assistant':
                    # There are some conversations where tool calls are not combined in one
                    if conv[j]['from'] == 'tool':
                        i = j
                        context += json.loads(conv[j]['value'])
                    j += 1
                else:
                    val = conv[j]['value']
                    if val.startswith('[') and val.endswith(']'): # Means the another tool is called
                        j += 1
                        continue
                    break
            else:
                return res
            new_entry['context'] = tool_responses_to_text(context)
            new_entry['output'] = conv[j]['value']
            res.append(new_entry)
        i += 1
    return res

def extract_tools_list_from_system(system_text: str):
    """
    Extract the *outer* JSON list of tool dicts from a ToolACE system prompt.

    Works even when the JSON contains nested [...] like enums/required, because it
    balances brackets and ignores brackets inside strings.
    """
    # Optional anchor: start searching after the "Here is a list..." line if present
    anchor = "Here is a list of functions"
    start_pos = system_text.find(anchor)
    if start_pos != -1:
        s = system_text[start_pos:]
    else:
        s = system_text

    # Find first '[' (start of the tool list)
    i0 = s.find("[")
    if i0 == -1:
        return None

    depth = 0
    in_str = False
    quote = ""
    esc = False
    start = None

    for i in range(i0, len(s)):
        ch = s[i]

        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == quote:
                in_str = False
            continue

        if ch in ('"', "'"):
            in_str = True
            quote = ch
            continue

        if ch == "[":
            if depth == 0:
                start = i
            depth += 1
        elif ch == "]":
            if depth > 0:
                depth -= 1
                if depth == 0 and start is not None:
                    block = s[start:i + 1]
                    try:
                        obj = json.loads(block)
                    except Exception:
                        return None

                    if isinstance(obj, list) and all(isinstance(x, dict) for x in obj) and any("name" in x for x in obj):
                        return obj
                    return None

    return None


## Parsing dataset

In [4]:
# List of dicts with keys 'query', 'context', 'output'
correct_dataset = []
all_tool_descriptions = []

for row in init_dataset.itertuples():
    system = row.system
    conv = row.conversations
    parsed_conv = parse_one_conversation(conv)
    parsed_tool_list = extract_tools_list_from_system(system)
    if parsed_tool_list and parsed_conv:
        keys_to_keep = ['name', 'description']
        parsed_tool_list = [{k:v for k, v in curr_tool.items() if k in keys_to_keep} for curr_tool in parsed_tool_list]
        all_tool_descriptions += [item['description'] for item in parsed_tool_list]
        for request in parsed_conv:
            old_context = request['context']
            new_context = f'{old_context}. Available tools: {json.dumps(parsed_tool_list)}.'
            request['context'] = new_context
    correct_dataset += parsed_conv

## Corrupting dataset

In [5]:
import json
from transformers import StoppingCriteria, StoppingCriteriaList

class SentenceStoppingCriteria(StoppingCriteria):
    def __init__(self, tokenizer, prompt_length):
        self.tokenizer = tokenizer
        self.prompt_length = prompt_length

    def __call__(self, input_ids, scores, **kwargs):

        # Decode only generated continuation
        generated_ids = input_ids[0][self.prompt_length:]

        text = self.tokenizer.decode(
            generated_ids,
            skip_special_tokens=True
        ).strip()

        return (len(text.split()) >= 5 and
         (text.endswith(".") or text.endswith("!") or text.endswith("?")))


model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
tokenizer = AutoTokenizer.from_pretrained(model_name)
generator = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    attn_implementation="eager",
).to('cuda')


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:
def corrupt(dataset, corruption_type, p=0.5):
    """Creates the corrupted dataset out of `dataset`.
    Supported corruption types are 'hallucination', 'overgeneration', 'missing_tool'.
    `p` is the probability of an entry of the `dataset` to be corrupted
    """
    corrupted_dataset = []
    for entry in tqdm(dataset):
        new_entry = copy.deepcopy(entry)
        if np.random.uniform(0, 1) < p:
            if corruption_type == 'missing_tool':
                suggest_call_to_missing_tool(new_entry)
            else:
                raise ValueError("Unknown hallucination type")
        else:
            new_entry['hallucination_labels'] = []
        corrupted_dataset.append(new_entry)
    return corrupted_dataset

def suggest_call_to_missing_tool(entry):
    available_tools = entry['context'].split('Available tools: ')[1]

    prompt = f"""
    You are modifying an assistant answer.

    Task:
    Generate exactly ONE additional sentence that:
    - sounds plausible and related to user request and original answer
    - asks the user if they want the assistant (you) to do something using the tool that is NOT present in this list:
    {available_tools}


    Rules:
    - Output ONLY the sentence.
    - Do NOT use quotation marks.
    - Do NOT add explanations.
    - Do NOT add multiple sentences.
    - Maximum 15 words.

    User request:
    {entry['query']}

    Original answer:
    {entry['output']}

    Additional sentence:
    """

    # Generate the response
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    stopping_criteria = StoppingCriteriaList([
        SentenceStoppingCriteria(
            tokenizer,
            prompt_length=inputs["input_ids"].shape[-1]
        )
    ])
    outputs = generator.generate(
        **inputs,
        max_new_tokens=100,
        min_new_tokens=5,
        temperature=0.7,
        do_sample=True,
        stopping_criteria=stopping_criteria,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

    # Decode and print
    sentence_to_add = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    init_len = len(entry['output'])
    entry['output'] = entry['output'] + ' ' + sentence_to_add
    entry['hallucination_labels'] = [{"start":init_len + 1,
                                      "end":init_len + 1 + len(sentence_to_add),
                                      "text":sentence_to_add,
                                      "type": "missing_tool"}]

In [10]:
hallucinated_dataset = corrupt(correct_dataset, 'missing_tool', 0.5)

  0%|          | 0/1034 [00:00<?, ?it/s]

In [11]:
with open("missing_tool_dataset.jsonl", "w", encoding="utf-8") as f:
    for sample in hallucinated_dataset:
        f.write(json.dumps(sample, ensure_ascii=False) + "\n")